In [ ]:
from sklearn.tree import DecisionTreeRegressor

import numpy as np

X = np.array([1., 2., 3., 4.]).reshape(-1, 1)
y = np.array([3., 5., 7., 9.])

learning_rate = 0.5

def forwardPass(y):
    return np.mean(y)

def MSE(y_pred, y_true): 
    return np.mean((y_pred - y_true)**2)

def residuals_func(y_true, y_pred):
    return y_true - y_pred

def main():
    y_pred = forwardPass(y)
    print("First predictions: ", y_pred)

    print("First MSE: ", MSE(y_pred, y))

    residuals = residuals_func(y, y_pred)
    print("Residuals: ", residuals)

    tree = DecisionTreeRegressor(
        max_depth=1,
        random_state=42
    )

    tree.fit(X, residuals)
    tree_predictions = tree.predict(X)

    print("Tree predictions: ", tree_predictions)

    new_predictions = y_pred + learning_rate * tree_predictions
    print("New predictions: ", new_predictions)

    new_mse = MSE(new_predictions, y)
    print("New MSE: ", new_mse)

if __name__ == "__main__":
    main()



First predictions:  6.0
First MSE:  5.0
Residuals:  [-3. -1.  1.  3.]
Tree predictions:  [-2. -2.  2.  2.]
New predictions:  [5. 5. 7. 7.]
New MSE:  2.0


#### Now i add a training loop

In [ ]:
from sklearn.tree import DecisionTreeRegressor

import numpy as np

X = np.array([1., 2., 3., 4.]).reshape(-1, 1)
y = np.array([3., 5., 7., 9.])

learning_rate = 0.5
n_estimators = 5

def forwardPass(y):
    return np.mean(y)

def MSE(y_pred, y_true): 
    return np.mean((y_pred - y_true)**2)

def residuals_func(y_true, y_pred):
    return y_true - y_pred

def predict(X, baseline, learning_rate, trees):
    baseline_arr = np.full(X.shape[0], baseline)

    tree_corrections = 0
    for tree in trees:
        tree_corrections += tree.predict(X)


    prediction = baseline_arr + learning_rate * tree_corrections
    return prediction

def main():
    trees = []

    y_pred = forwardPass(y)
    baseline = y_pred
    print("First predictions: ", y_pred)

    print("First MSE: ", MSE(y_pred, y))

    for i in range(n_estimators):
        residuals = residuals_func(y, y_pred)
        print("Residuals: ", residuals.round(3))

        tree = DecisionTreeRegressor(
            max_depth=1,
            random_state=42
        )

        tree.fit(X, residuals)
        tree_predictions = tree.predict(X)

        trees.append(tree)

        print(f"Tree {i} predictions: ", tree_predictions.round(3))

        y_pred = y_pred + learning_rate * tree_predictions
        print("New predictions: ", y_pred.round(3))

        new_mse = MSE(y_pred, y)
        print(f"New {i} MSE: ", new_mse)


    X_new = np.array([[2.8]])

    prediction = predict(X_new, baseline, learning_rate, trees)

    print("Prediction:", prediction)

if __name__ == "__main__":
    main()



First predictions:  6.0
First MSE:  5.0
Residuals:  [-3. -1.  1.  3.]
Tree 0 predictions:  [-2. -2.  2.  2.]
New predictions:  [5. 5. 7. 7.]
New 0 MSE:  2.0
Residuals:  [-2.  0.  0.  2.]
Tree 1 predictions:  [-2.     0.667  0.667  0.667]
New predictions:  [4.    5.333 7.333 7.333]
New 1 MSE:  1.0
Residuals:  [-1.    -0.333 -0.333  1.667]
Tree 2 predictions:  [-0.556 -0.556 -0.556  1.667]
New predictions:  [3.722 5.056 7.056 8.167]
New 2 MSE:  0.3055555555555558
Residuals:  [-0.722 -0.056 -0.056  0.833]
Tree 3 predictions:  [-0.278 -0.278 -0.278  0.833]
New predictions:  [3.583 4.917 6.917 8.583]
New 3 MSE:  0.13194444444444478
Residuals:  [-0.583  0.083  0.083  0.417]
Tree 4 predictions:  [-0.583  0.194  0.194  0.194]
New predictions:  [3.292 5.014 7.014 8.681]
New 4 MSE:  0.04687500000000007
Prediction: [7.01388889]


### Real dataset - diabetes

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
import numpy as np

X, y = load_diabetes(return_X_y=True)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.25, random_state=42
)

def forwardPass(y):
    return np.mean(y)

def MSE(y_pred, y_true): 
    return np.mean((y_pred - y_true)**2)

def residuals_func(y_true, y_pred):
    return y_true - y_pred

def predict(X, baseline, learning_rate, trees):
    baseline_arr = np.full(X.shape[0], baseline)

    tree_corrections = 0
    for tree in trees:
        tree_corrections += tree.predict(X)


    prediction = baseline_arr + learning_rate * tree_corrections
    return prediction

def fit(X, y, learning_rate, n_estimators, max_depth):
    trees = []

    y_pred = forwardPass(y)
    baseline = y_pred

#    print("First predictions: ", y_pred)
#
#    print("First MSE: ", MSE(y_pred, y))

    for _ in range(n_estimators):
        residuals = residuals_func(y, y_pred)
#        print("Residuals: ", residuals.round(3))

        tree = DecisionTreeRegressor(
            max_depth=max_depth,
            random_state=42
        )

        tree.fit(X, residuals)
        tree_predictions = tree.predict(X)

        trees.append(tree)

#        print(f"Tree {i} predictions: ", tree_predictions.round(3))
#
        y_pred = y_pred + learning_rate * tree_predictions
#        print("New predictions: ", y_pred.round(3))
#
#        new_mse = MSE(y_pred, y)
#        print(f"New {i} MSE: ", new_mse)


    return baseline, trees


for i in range(3):
    learning_rate = [0.01, 0.1, 0.5]
    n_estimators = 100
    max_depth = 2

    print(f"Model nr: {i+1}, learnign_rate: {learning_rate[i]}")
    baseline, trees = fit(X_train, y_train, learning_rate[i], n_estimators, max_depth)

    train_pred = predict(X_train, baseline, learning_rate[i], trees)
    training_mse = MSE(train_pred, y_train)

    validation_pred = predict(X_val, baseline, learning_rate[i], trees)
    validation_mse = MSE(validation_pred, y_val)

    print("Train MSE: ", training_mse)
    print("Validation MSE: ", validation_mse)



#    X_new = np.array([[2.8]])
#
#    prediction = predict(X_new, baseline, learning_rate, trees)
#
#    print("Prediction:", prediction)

Model nr: 1, learnign_rate: 0.01
Train MSE:  3447.1067276871954
Validation MSE:  3385.1650045110036
Model nr: 2, learnign_rate: 0.1
Train MSE:  1459.8972749542263
Validation MSE:  2886.9293330083597
Model nr: 3, learnign_rate: 0.5
Train MSE:  250.91720939449277
Validation MSE:  4423.902353200871


### Gradient Boosting Classifier

In [11]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

learning_rate = 0.5
n_estimators = 5
max_depth = 1

X = np.array([1., 2., 3., 4.]).reshape(-1, 1)
y = np.array([0., 0., 1., 1.])

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def BCE(y, p):
    return np.mean(-(y*np.log(p) + (1-y)*np.log(1-p)))

def initial_prediction(y):
    y_mean = np.mean(y)

    logit = np.log(y_mean / (1-y_mean))

    return logit

def residuals_func(y, p):
    return y-p

def fit(X,y, learning_rate, n_estimators, max_depth):
    baseline = initial_prediction(y)

    logits = np.full(y.shape, baseline, dtype=float)

    trees = []

    for i in range(n_estimators):
        y_pred = sigmoid(logits)

        r = residuals_func(y, y_pred)

        tree = DecisionTreeRegressor(
            max_depth=max_depth,
            random_state=42
        )

        tree.fit(X, r)

        tree_logits_corrections = tree.predict(X)
        print("Tree predictions: ", tree_logits_corrections)

        logits += learning_rate * tree_logits_corrections

        print("New logits: ", logits)

        trees.append(tree)

        new_pred = sigmoid(logits)
        print("Probabilities: ", new_pred)

        print(f"BCE nr.{i}: ", BCE(y, new_pred))

    return baseline, trees


fit(X, y, learning_rate, n_estimators, max_depth)

#baseline = initial_prediction(y)
#initial_logits = np.full(y.shape, baseline, dtype=float)
#initial_probabilities = sigmoid(initial_logits)
#
#print("Baseline:", baseline)
#print("Initial logits:", initial_logits)
#print("Initial probabilities:", initial_probabilities)
#print("Initial BCE:", BCE(y, initial_probabilities))
#print("Residuals:", residuals_func(y, initial_probabilities))

Tree predictions:  [-0.5 -0.5  0.5  0.5]
New logits:  [-0.25 -0.25  0.25  0.25]
Probabilities:  [0.4378235 0.4378235 0.5621765 0.5621765]
BCE nr.0:  0.5759394198788437
Tree predictions:  [-0.4378235 -0.4378235  0.4378235  0.4378235]
New logits:  [-0.46891175 -0.46891175  0.46891175  0.46891175]
Probabilities:  [0.38487385 0.38487385 0.61512615 0.61512615]
BCE nr.1:  0.4859279106088488
Tree predictions:  [-0.38487385 -0.38487385  0.38487385  0.38487385]
New logits:  [-0.66134867 -0.66134867  0.66134867  0.66134867]
Probabilities:  [0.34043672 0.34043672 0.65956328 0.65956328]
BCE nr.2:  0.41617735327557165
Tree predictions:  [-0.34043672 -0.34043672  0.34043672  0.34043672]
New logits:  [-0.83156703 -0.83156703  0.83156703  0.83156703]
Probabilities:  [0.30331383 0.30331383 0.69668617 0.69668617]
BCE nr.3:  0.3614202298631523
Tree predictions:  [-0.30331383 -0.30331383  0.30331383  0.30331383]
New logits:  [-0.98322395 -0.98322395  0.98322395  0.98322395]
Probabilities:  [0.27225255 0.2

(np.float64(0.0),
 [DecisionTreeRegressor(max_depth=1, random_state=42),
  DecisionTreeRegressor(max_depth=1, random_state=42),
  DecisionTreeRegressor(max_depth=1, random_state=42),
  DecisionTreeRegressor(max_depth=1, random_state=42),
  DecisionTreeRegressor(max_depth=1, random_state=42)])